# ObjectOracle — Quick Demo Notebook
Run detection locally without the API server.

In [ ]:
import sys
sys.path.insert(0, '../api')

from PIL import Image
import requests
from io import BytesIO

# Load a sample image
url = 'https://ultralytics.com/images/bus.jpg'
img = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
img

In [ ]:
from core.model_loader import registry
registry.initialize()

In [ ]:
from core.inference import run_full_pipeline
import json

result = run_full_pipeline(img, include_scene=True, include_segmentation=False)
print(f"Inference: {result['inference_ms']}ms")
print(f"Objects:   {len(result['objects'])}")
print(f"Scene:     {result['scene_tags'][0]['label']}")
print(json.dumps(result, indent=2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(img)

colors = plt.cm.tab10.colors
for i, obj in enumerate(result['objects']):
    x1, y1, x2, y2 = obj['bbox']
    color = colors[obj['class_id'] % 10]
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                               linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, y1-5, f"{obj['label']} {obj['confidence']:.2f}",
            color='white', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

ax.set_title(f"ObjectOracle — {len(result['objects'])} objects | {result['inference_ms']}ms", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()